# Demo — RAG híbrido: palavra-chave e vetor no mesmo índice

**Objetivo:** aplicar às seis perguntas do slide 14 ("O primeiro degrau: por que busca vetorial pura não basta") o remédio que o próprio slide propõe: a busca híbrida, em que o BM25 (palavra-chave) e o vetor consultam o mesmo índice, na mesma requisição, e o RRF funde as duas listas. É a linha **Híbrido** do slide 15 (termo exato, sigla, código e número, por um custo "quase o mesmo") e o degrau 1 do slide 17.

1. **Como o índice enxerga códigos** → o analisador `pt-BR.lucene` quebra "SKU 48213-B", "NR-12 rev. 3" e "projeto ATLAS" em tokens;
2. **Três consultas, o mesmo índice** → texto (BM25), vetor e híbrido nas seis perguntas do slide 14;
3. **Lupa na string exata** → o top 5 de cada consulta para o SKU 48213-B, lado a lado, e o que é o score do RRF;
4. **RAG completo** → a resposta com citação a partir dos 3 primeiros trechos da busca híbrida.

> **Cenário 100% fictício.** A Contoso Industrial e todos os documentos, projetos, pessoas e números do `acervo/` existem apenas para fins didáticos.

**Pré-requisito:** o índice `cap05-acervo`, criado e carregado pelo notebook `tipos-de-rag/00-indice-compartilhado`.

## 0. Preparação

Variáveis necessárias:

- `AZURE_OPENAI_ENDPOINT` e `AZURE_OPENAI_API_KEY`
- `AZURE_OPENAI_DEPLOYMENT` (deployment de chat, usado na Execução 4)
- `AZURE_OPENAI_EMBEDDING_DEPLOYMENT`, `AZURE_OPENAI_EMBEDDING_MODEL` e `AZURE_OPENAI_EMBEDDING_DIMENSIONS`: os **mesmos** valores usados para criar o índice no notebook 00
- `AZURE_SEARCH_ENDPOINT` e `AZURE_SEARCH_API_KEY`

Os valores vêm do arquivo `Cap05/.env` (copie o `.env.example` e preencha); o que faltar é perguntado na hora da execução.

> Não grave chaves diretamente no notebook.

In [ ]:
%pip install -q "openai>=2" "python-dotenv>=1.0" "pandas>=2.1" "azure-search-documents>=12.0.0,<13"

In [ ]:
import os
from getpass import getpass
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

load_dotenv(find_dotenv(usecwd=True))  # lê o .env da pasta Cap05, se existir

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT") or input(
    "AZURE_OPENAI_ENDPOINT (ex.: https://meu-recurso.openai.azure.com): "
).strip()
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY") or getpass("AZURE_OPENAI_API_KEY: ")
AZURE_OPENAI_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT") or input(
    "AZURE_OPENAI_DEPLOYMENT (deployment de chat): "
).strip()
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT") or input(
    "AZURE_OPENAI_EMBEDDING_DEPLOYMENT (deployment de embedding): "
).strip()

# O modelo e a dimensão do embedding são o contrato entre indexação e consulta.
EMBEDDING_MODEL = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")
EMBEDDING_DIMENSIONS = int(os.getenv("AZURE_OPENAI_EMBEDDING_DIMENSIONS", "1536"))

client = OpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    base_url=f"{AZURE_OPENAI_ENDPOINT.rstrip('/')}/openai/v1/",
)

# Pasta Cap05: a primeira pasta, subindo a partir daqui, que contém o acervo.
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "acervo").is_dir())
PASTA_ACERVO = RAIZ / "acervo"

print("Cliente configurado.")
print("Deployment de chat:", AZURE_OPENAI_DEPLOYMENT)
print("Embedding:", AZURE_OPENAI_EMBEDDING_DEPLOYMENT, f"({EMBEDDING_MODEL}, {EMBEDDING_DIMENSIONS} dimensões)")
print("Acervo:", PASTA_ACERVO)

In [ ]:
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient

AZURE_SEARCH_ENDPOINT = os.getenv("AZURE_SEARCH_ENDPOINT") or input(
    "AZURE_SEARCH_ENDPOINT (ex.: https://meu-servico.search.windows.net): "
).strip()
AZURE_SEARCH_API_KEY = os.getenv("AZURE_SEARCH_API_KEY") or getpass("AZURE_SEARCH_API_KEY (chave de administração): ")

NOME_INDICE = "cap05-acervo"
credencial_busca = AzureKeyCredential(AZURE_SEARCH_API_KEY)
index_client = SearchIndexClient(AZURE_SEARCH_ENDPOINT, credencial_busca)
search_client = SearchClient(AZURE_SEARCH_ENDPOINT, NOME_INDICE, credencial_busca)

# Este notebook só consulta: quem cria e carrega o índice é o notebook 00.
if NOME_INDICE not in list(index_client.list_index_names()):
    raise RuntimeError("Índice não encontrado. Rode antes o notebook tipos-de-rag/00-indice-compartilhado.")

print(f"Índice '{NOME_INDICE}' encontrado, com {search_client.get_document_count()} trechos.")

### Funções auxiliares

- `gerar_embeddings`: a mesma dos notebooks 00 e 01, com o mesmo deployment e a mesma dimensão. A pergunta precisa cair no mesmo espaço vetorial dos trechos.
- `responder`: a geração com citação, usada na Execução 4.
- `PERGUNTAS` e `posicao`: o gabarito (`acervo/perguntas.json`) diz qual trecho responde cada pergunta; `posicao` diz em que lugar ele apareceu.

In [ ]:
pd.set_option("display.max_colwidth", None)  # tabelas mostram o texto inteiro


def gerar_embeddings(textos: list[str], lote: int = 16) -> list[list[float]]:
    vetores = []
    for i in range(0, len(textos), lote):
        resposta = client.embeddings.create(
            model=AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
            input=textos[i:i + lote],
            dimensions=EMBEDDING_DIMENSIONS,
        )
        vetores.extend(item.embedding for item in resposta.data)
    return vetores


INSTRUCOES_RAG = """
Responda em português, exclusivamente com base nos trechos fornecidos.
Cite a fonte de cada afirmação no formato [id do trecho].
Se os trechos não trouxerem a informação, responda:
'Não há evidência suficiente nos trechos recuperados.'
""".strip()


def responder(pergunta: str, trechos_recuperados: list[dict]) -> str:
    contexto = "\n\n".join(
        f"[{t['id']}] {t['titulo']} — {t['secao']}\n{t['conteudo']}" for t in trechos_recuperados
    )
    resposta = client.responses.create(
        model=AZURE_OPENAI_DEPLOYMENT,
        instructions=INSTRUCOES_RAG,
        input=f"TRECHOS RECUPERADOS:\n{contexto}\n\nPERGUNTA:\n{pergunta}",
        store=False,
    )
    return resposta.output_text

In [ ]:
import json

PERGUNTAS = {p["id"]: p for p in json.loads((PASTA_ACERVO / "perguntas.json").read_text(encoding="utf-8"))}

# As seis perguntas do slide 14: três em que o vetor acerta e três em que ele erra.
PERGUNTAS_SLIDE_14 = ["linguagem-natural", "parafrase", "assunto-certo", "codigo-de-norma", "sigla-interna", "string-exata"]


def posicao(ids_retornados: list[str], grupo: list[str]) -> int | None:
    """Posição (1, 2, 3...) do primeiro trecho esperado na lista retornada, ou None se ele não apareceu."""
    for i, id_trecho in enumerate(ids_retornados, start=1):
        if id_trecho in grupo:
            return i
    return None

---

# Execução 1 — Como o índice enxerga códigos

A busca por palavra-chave não compara a pergunta com o texto do jeito que ele foi escrito: compara **tokens**. No notebook 00, os campos `titulo`, `secao` e `conteudo` foram criados pesquisáveis, com o analisador `pt-BR.lucene`. É ele que quebra cada trecho em tokens na indexação, e quebra a pergunta do mesmo jeito na hora da consulta.

A API `analyze_text` mostra essa quebra para qualquer texto, sem fazer busca nenhuma.

In [ ]:
from azure.search.documents.indexes.models import AnalyzeTextOptions, LexicalAnalyzerName

linhas = []
for texto in ["SKU 48213-B", "NR-12 rev. 3", "projeto ATLAS"]:
    analise = index_client.analyze_text(
        NOME_INDICE,
        AnalyzeTextOptions(text=texto, analyzer_name=LexicalAnalyzerName.PT_BR_LUCENE),
    )
    tokens = [t.token for t in analise.tokens]
    linhas.append({"texto original": texto, "tokens gerados": " · ".join(tokens), "quantos tokens": len(tokens)})

display(Markdown("**Analisador:** `pt-BR.lucene`, o mesmo dos campos `titulo`, `secao` e `conteudo`"))
display(pd.DataFrame(linhas))

### O que observar

- O hífen separa o código em dois tokens: "48213-B" vira `48213` e `b`. Tudo vai para minúsculas, e a pontuação ("rev.") fica de fora.
- O BM25 compara esses tokens **exatamente**: as seções do 48213-C e do 48213-D têm o `48213`, mas não têm o `b`. O embedding só **aproxima**: para ele, "48213-B" e "48213-C" são textos quase iguais (slide 14).
- O analisador de português também corta o fim das palavras (stemming), para que "projeto" e "projetos" casem. A forma do token não importa: a pergunta passa pelo mesmo analisador e vira os mesmos tokens do índice.

### Leitura arquitetural

- O analisador é parte do contrato do índice: vale para a indexação e para a consulta, e trocar o analisador de um campo obriga a recriar o índice.
- Um detalhe que o acervo evita de propósito: "a" é stopword no `pt-BR.lucene` e some na análise. Um "SKU 48213-A" perderia justamente a letra que o diferencia.

---

# Execução 2 — Três consultas, o mesmo índice

A mesma pergunta vai ao mesmo índice de três jeitos. Só muda o que vai na requisição:

- **texto (BM25):** só `search_text`. O BM25 pontua os trechos que têm pelo menos um token da pergunta, e dá mais peso aos tokens raros no acervo.
- **vetor:** só `vector_queries`, com a mesma `buscar_vetorial` do notebook 01.
- **híbrido:** `search_text` **e** `vector_queries` na mesma chamada. O AI Search roda as duas buscas em paralelo e funde as duas listas com o RRF (Reciprocal Rank Fusion).

Na busca híbrida, o vetor entrega `k_nearest_neighbors=50` candidatos para a fusão (a documentação sugere começar com 30 a 50, e 50 é o que o semantic ranker do notebook 03 vai precisar). A função devolve os `k=5` primeiros da lista fundida.

### O que esperar e por que pode variar

- O texto tende a ganhar nos códigos e a perder nas paráfrases: "férias" não divide nenhuma palavra com "descanso remunerado". Quando o texto acerta uma paráfrase, é de carona em palavras que a pergunta e o trecho dividem por acaso ("dias", "ferramenta"), não porque o BM25 entendeu o assunto.
- O vetor tende ao contrário: acerta paráfrase e sinônimo, e tropeça em código, sigla e string exata (notebook 01).
- O híbrido tende a acertar os dois lados, porque o RRF premia o trecho bem colocado em qualquer uma das duas listas.
- Os números mudam com o modelo de embedding, com a redação da pergunta e com o próprio acervo (o peso de cada palavra no BM25 depende de quantos trechos a contêm): não trate a tabela como gabarito fixo.
- Num acervo pequeno como este, as três colunas podem coincidir, com 1º em quase tudo: o vetor pode acertar um código por sorte, e o texto, uma paráfrase de carona. Nesse caso, a lição está nos scores, e não na posição. Na lupa da Execução 3, compare os scores dentro de cada coluna: o BM25 tende a separar o 48213-B dos irmãos 48213-C e 48213-D com folga, e o vetor tende a deixar os três quase empatados.

In [ ]:
from azure.search.documents.models import VectorizedQuery


def buscar_texto(pergunta: str, k: int = 5) -> list[dict]:
    """Busca por palavra-chave (BM25): só o texto da pergunta, sem vetor."""
    resultados = search_client.search(
        search_text=pergunta,
        select=["id", "titulo", "secao", "conteudo", "fonte"],
        top=k,
    )
    return [
        {
            "id": r["id"],
            "titulo": r["titulo"],
            "secao": r["secao"],
            "conteudo": r["conteudo"],
            "fonte": r["fonte"],
            "score": r["@search.score"],
        }
        for r in resultados
    ]


def buscar_vetorial(pergunta: str, k: int = 5) -> list[dict]:
    """Busca vetorial pura: vetoriza a pergunta no cliente e pede ao índice os k vizinhos mais próximos."""
    vetor = gerar_embeddings([pergunta])[0]
    resultados = search_client.search(
        search_text=None,
        vector_queries=[VectorizedQuery(vector=vetor, k_nearest_neighbors=k, fields="embedding")],
        select=["id", "titulo", "secao", "conteudo", "fonte"],
        top=k,
    )
    return [
        {
            "id": r["id"],
            "titulo": r["titulo"],
            "secao": r["secao"],
            "conteudo": r["conteudo"],
            "fonte": r["fonte"],
            "score": r["@search.score"],
        }
        for r in resultados
    ]


def buscar_hibrida(pergunta: str, k: int = 5) -> list[dict]:
    """Busca híbrida: o texto e o vetor da pergunta na mesma requisição; devolve os k primeiros da fusão (RRF)."""
    vetor = gerar_embeddings([pergunta])[0]
    resultados = search_client.search(
        search_text=pergunta,
        vector_queries=[VectorizedQuery(vector=vetor, k_nearest_neighbors=50, fields="embedding")],
        select=["id", "titulo", "secao", "conteudo", "fonte"],
        top=k,
    )
    return [
        {
            "id": r["id"],
            "titulo": r["titulo"],
            "secao": r["secao"],
            "conteudo": r["conteudo"],
            "fonte": r["fonte"],
            "score": r["@search.score"],
        }
        for r in resultados
    ]

In [ ]:
BUSCAS = {"texto": buscar_texto, "vetor": buscar_vetorial, "híbrido": buscar_hibrida}

resultados_slide_14 = {
    pid: {modo: buscar(PERGUNTAS[pid]["pergunta"], k=5) for modo, buscar in BUSCAS.items()}
    for pid in PERGUNTAS_SLIDE_14
}

linhas = []
for pid, por_modo in resultados_slide_14.items():
    p = PERGUNTAS[pid]
    linha = {"pergunta": p["pergunta"], "categoria": p["categoria"]}
    for modo, top5 in por_modo.items():
        lugar = posicao([t["id"] for t in top5], p["trechos_esperados"][0])
        linha[modo] = f"{lugar}º" if lugar else "fora do top 5"
    linhas.append(linha)

display(Markdown("**Posição do trecho certo no top 5 de cada consulta**"))
display(pd.DataFrame(linhas))

### O que observar

- Leia a tabela por linha: em qual coluna o trecho certo sobe e em qual ele cai. Se as três colunas coincidirem, procure a diferença nos scores da lupa (Execução 3).
- Nas três últimas linhas, o texto tende a acertar: pergunta e trecho têm o mesmo token raro (o `12` da NR-12, o nome ATLAS, o `b` do SKU). Nas três primeiras, o vetor tende a acertar: pergunta e trecho falam do mesmo assunto, ainda que as palavras mudem (paráfrase e sinônimo).
- A coluna **híbrido** tende a ficar boa nas seis. Não é garantia: o RRF olha só a posição, e um trecho em 1º numa lista e longe na outra pode perder para um que veio em 2º nas duas.

---

# Execução 3 — Lupa na string exata

Abrimos o top 5 das três consultas para a pergunta do SKU 48213-B, com a seção e o `@search.score` de cada trecho. Os três scores **não estão na mesma régua**:

- **texto:** score do BM25, sem teto. Cresce com os tokens em comum, e mais ainda com os tokens raros.
- **vetor:** de 0,333 a 1 (cosseno), como no notebook 01.
- **híbrido:** score do RRF. Cada lista dá 1/60 ao 1º, 1/61 ao 2º, e assim por diante (é o `1/(60 + rank)` da documentação, com o rank contado a partir de zero), e o trecho soma o que recebeu das duas listas. Quem fica em 1º nas duas chega a 2/60, cerca de 0,033. Por isso os números são pequenos: um 0,03 no híbrido pode ser um ótimo resultado. Eles também ficam muito próximos entre si: a lupa mostra o híbrido com 6 casas decimais porque, com 4, scores diferentes pareceriam iguais.

O 60 é uma constante do RRF e não tem nada a ver com o `k_nearest_neighbors=50` nem com o `k=5` das nossas funções.

In [ ]:
pid = "string-exata"
esperados = PERGUNTAS[pid]["trechos_esperados"][0]

# O híbrido ganha 6 casas decimais: os scores do RRF ficam muito próximos entre si.
lado_a_lado = pd.concat(
    {
        modo: pd.DataFrame(
            [
                {
                    "seção": t["secao"] + (" ✔" if t["id"] in esperados else ""),
                    "score": round(t["score"], 6 if modo == "híbrido" else 4),
                }
                for t in top5
            ],
            index=range(1, len(top5) + 1),
        )
        for modo, top5 in resultados_slide_14[pid].items()
    },
    axis=1,
)

display(Markdown(f"**{PERGUNTAS[pid]['pergunta']}** (✔ = o trecho certo)"))
display(lado_a_lado.rename_axis("posição"))

### O que observar

- **texto:** a seção "SKU 48213-B" do catálogo tende a vir em 1º, porque junta `48213`, `b`, "prazo" e "entrega". Outros trechos que citam o 48213-B (fornecedores, plano de manutenção) e os irmãos 48213-C e 48213-D, que têm o `48213`, vêm atrás.
- **vetor:** os três SKUs tendem a aparecer juntos, com scores quase iguais (a lupa do notebook 01).
- **híbrido:** o 48213-B tende a ficar no topo, porque está bem colocado nas duas listas. A ordem final vem das posições nas duas listas, não dos scores originais.
- Não compare score entre colunas: um 8,5 no texto não é "melhor" que um 0,032 no híbrido. Cada algoritmo tem a sua escala; entre colunas, o que se compara é a **posição**.

---

# Execução 4 — RAG completo

Os 3 primeiros trechos da busca híbrida vão para o modelo, com a mesma `responder` do notebook 01. A pergunta é a da string exata, justamente a que o vetor puro tende a errar.

### O que esperar e por que pode variar

- Se o trecho do 48213-B estiver entre os três enviados, a resposta tende a trazer o prazo certo (18 dias úteis), citando o `id` do trecho.
- Se ele ficar de fora, o erro continua sendo da recuperação, não do modelo (slide 29). Leia a linha **O trecho certo está entre eles?** antes de julgar a resposta.

In [ ]:
pid = "string-exata"
pergunta = PERGUNTAS[pid]["pergunta"]
top3 = resultados_slide_14[pid]["híbrido"][:3]
lugar = posicao([t["id"] for t in top3], PERGUNTAS[pid]["trechos_esperados"][0])
enviados = ", ".join(f"`{t['id']}`" for t in top3)
situacao = f"sim, em {lugar}º lugar" if lugar else "não"

display(Markdown(f"### {pergunta}"))
display(Markdown(f"**Trechos enviados ao modelo:** {enviados}  \n**O trecho certo está entre eles?** {situacao}"))
display(Markdown(responder(pergunta, top3)))

### O que observar

- Compare com a Execução 4 do notebook 01: a mesma pergunta, o mesmo modelo, o mesmo prompt. Só a recuperação mudou.
- Os irmãos 48213-C e 48213-D podem vir junto no contexto. A instrução de responder só com base nos trechos, citando o `id`, ajuda o modelo a não misturar os prazos, e a citação permite conferir.

### Leitura arquitetural

- **Custo "quase o mesmo"** (slide 15): o mesmo índice, uma única requisição, o mesmo vetor da pergunta. O BM25 já estava lá, porque os campos de texto foram criados pesquisáveis no notebook 00. Ligar o híbrido é mandar `search_text` junto com o vetor.
- É o ponto de partida padrão: "Comece em híbrido com reranking" (slide 13). E o primeiro anti-padrão do slide 28, vetorial puro em acervo cheio de código, sigla e número, se resolve com "Híbrido desde o primeiro dia".
- O próximo degrau do slide 17: quando o assunto certo vem, mas o trecho que responde fica fora dos primeiros, entra o semantic ranker sobre os 50 candidatos do híbrido. É o notebook 03.

### Fala sugerida

> “Comece em híbrido com reranking. Suba para agêntico ou grafo por sintoma medido, nunca por moda.”

---

# Quadro final — o que cada execução provou?

| Execução | O que vimos | Decisão arquitetural |
|---|---|---|
| 1 | O analisador quebra "48213-B" em `48213` + `b`, e a pergunta passa pelo mesmo analisador | O analisador dos campos de texto é contrato do índice: escolher na criação |
| 2 | O texto tende a acertar código, sigla e número; o vetor, paráfrase e sinônimo; o híbrido, os dois | Híbrido como padrão: BM25 e vetor no mesmo índice, na mesma requisição |
| 3 | O score do híbrido é do RRF: pequeno, e só serve para ordenar | Comparar posições, nunca scores de tipos de busca diferentes |
| 4 | Com o trecho certo no contexto, a resposta acerta e cita a fonte | Subir para reranking (notebook 03) quando o trecho certo vier fora dos primeiros |

## Frase de fechamento

> **Remédio: busca híbrida. O BM25 no mesmo índice acha a string exata que o vetor aproxima.**

---

# Referências oficiais

- Azure AI Search — busca híbrida: visão geral:  
  https://learn.microsoft.com/azure/search/hybrid-search-overview

- Azure AI Search — criar uma consulta híbrida:  
  https://learn.microsoft.com/azure/search/hybrid-search-how-to-query

- Azure AI Search — o score da busca híbrida (RRF):  
  https://learn.microsoft.com/azure/search/hybrid-search-ranking

- Azure AI Search — analisadores de texto e a API Analyze:  
  https://learn.microsoft.com/azure/search/search-analyzers